In [47]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import *
import pandas as pd

In [48]:
spark = SparkSession.builder.appName("Q1").getOrCreate()

In [49]:
df = spark.read.csv("emp_data.csv",header=True,inferSchema=True)
df.show()

+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|EmpID|FirstName| LastName|StartDate| ExitDate|             Title|      Supervisor|BusinessUnit|EmployeeStatus|EmployeeType|PayZone|EmployeeClassificationType|TerminationType|DepartmentType|            Division|       DOB|State|JobFunctionDescription|GenderCode|LocationCode|RaceDesc|MaritalDesc|Performance Score|Current Employee Rating|
+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+--------

In [50]:
df.printSchema()

root
 |-- EmpID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- StartDate: string (nullable = true)
 |-- ExitDate: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Supervisor: string (nullable = true)
 |-- BusinessUnit: string (nullable = true)
 |-- EmployeeStatus: string (nullable = true)
 |-- EmployeeType: string (nullable = true)
 |-- PayZone: string (nullable = true)
 |-- EmployeeClassificationType: string (nullable = true)
 |-- TerminationType: string (nullable = true)
 |-- DepartmentType: string (nullable = true)
 |-- Division: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- State: string (nullable = true)
 |-- JobFunctionDescription: string (nullable = true)
 |-- GenderCode: string (nullable = true)
 |-- LocationCode: integer (nullable = true)
 |-- RaceDesc: string (nullable = true)
 |-- MaritalDesc: string (nullable = true)
 |-- Performance Score: string (nullable = true)
 |-- Cu

In [51]:
df.select([count(when(isnull(c)|isnan(c),c)).alias(c) for c in df.columns]).show()

+-----+---------+--------+---------+--------+-----+----------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------+---+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|EmpID|FirstName|LastName|StartDate|ExitDate|Title|Supervisor|BusinessUnit|EmployeeStatus|EmployeeType|PayZone|EmployeeClassificationType|TerminationType|DepartmentType|Division|DOB|State|JobFunctionDescription|GenderCode|LocationCode|RaceDesc|MaritalDesc|Performance Score|Current Employee Rating|
+-----+---------+--------+---------+--------+-----+----------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------+---+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|    0|        0|       0|        0|     263|    0|         0|           0|             0|           0|

In [52]:
df = df.withColumn("LastName",when(isnull(col("LastName")),"UnKnown").otherwise(col("LastName")))

In [53]:
df = df.filter(col("EmpID").isNotNull() & col("StartDate").isNotNull())

In [54]:
df = df.withColumn("Current Employee Rating",
                   when(col("Current Employee Rating")<1,1).
                   when(col("Current Employee Rating")>5,5).
                   otherwise(col("Current Employee Rating")))
df.show()

+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|EmpID|FirstName| LastName|StartDate| ExitDate|             Title|      Supervisor|BusinessUnit|EmployeeStatus|EmployeeType|PayZone|EmployeeClassificationType|TerminationType|DepartmentType|            Division|       DOB|State|JobFunctionDescription|GenderCode|LocationCode|RaceDesc|MaritalDesc|Performance Score|Current Employee Rating|
+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+--------

In [58]:
df = df.filter((length(col("LocationCode"))==5)&(col("LocationCode").cast("int").isNotNull()))
df.show()

+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|EmpID|FirstName| LastName|StartDate| ExitDate|             Title|      Supervisor|BusinessUnit|EmployeeStatus|EmployeeType|PayZone|EmployeeClassificationType|TerminationType|DepartmentType|            Division|       DOB|State|JobFunctionDescription|GenderCode|LocationCode|RaceDesc|MaritalDesc|Performance Score|Current Employee Rating|
+-----+---------+---------+---------+---------+------------------+----------------+------------+--------------+------------+-------+--------------------------+---------------+--------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+--------

In [59]:
df = df.dropDuplicates()
df.show()

+-----+---------+---------+---------+---------+--------------------+-----------------+------------+--------------------+------------+-------+--------------------------+---------------+-----------------+--------------------+----------+-----+----------------------+----------+------------+--------+-----------+-----------------+-----------------------+
|EmpID|FirstName| LastName|StartDate| ExitDate|               Title|       Supervisor|BusinessUnit|      EmployeeStatus|EmployeeType|PayZone|EmployeeClassificationType|TerminationType|   DepartmentType|            Division|       DOB|State|JobFunctionDescription|GenderCode|LocationCode|RaceDesc|MaritalDesc|Performance Score|Current Employee Rating|
+-----+---------+---------+---------+---------+--------------------+-----------------+------------+--------------------+------------+-------+--------------------------+---------------+-----------------+--------------------+----------+-----+----------------------+----------+------------+--------+--

In [60]:
designation_count = df.groupBy(trim("DepartmentType"),trim("Title")).count()
designation_count.show()

+--------------------+--------------------+-----+
|trim(DepartmentType)|         trim(Title)|count|
+--------------------+--------------------+-----+
|               IT/IS|Principal Data Ar...|    7|
|               IT/IS|        Data Analyst|    4|
|          Production|Production Techni...|   54|
|               IT/IS|Sr. Network Engineer|   25|
|    Executive Office|    Network Engineer|   15|
|               IT/IS|    Network Engineer|   17|
|          Production|Production Techni...|    1|
|          Production|Production Techni...|  157|
|               IT/IS|      Data Architect|    2|
|               IT/IS|  Area Sales Manager|    4|
|               Sales|  Area Sales Manager|  106|
|               IT/IS|Enterprise Architect|    4|
|               IT/IS|          IT Support|   36|
|               Sales| Area Sales Manager?|    1|
|               Sales| Area Sales Manager.|    1|
|               IT/IS|Software Engineer...|    1|
|               IT/IS|      Sr. Accountant|    4|


In [61]:
high_performance = df.groupBy(trim("DepartmentType")).agg(max("Current Employee Rating").alias("max_performance"))
high_performance.show()

+--------------------+---------------+
|trim(DepartmentType)|max_performance|
+--------------------+---------------+
|               Sales|              5|
|          Production|              5|
|    Executive Office|              3|
|               IT/IS|              4|
+--------------------+---------------+



In [62]:
pandas_df = df.toPandas()
pandas_df.to_csv("cleaned_emp.csv",index=False)